# Needle End-to-End Training Entry Notebook

This notebook is a **practical project entrypoint** for the Needle homework codebase.
It is intentionally different from `hw4.ipynb`: instead of grading-oriented checks, this notebook focuses on **real training and inference workflows** through `apps/`.

## What this notebook does

1. Prepares runtime paths so `apps` and `python/needle` are importable.
2. Tries to build the C++ backend (`nd`) and falls back to numpy backend (`np`) when unavailable.
3. Downloads datasets:
   - CIFAR-10 for vision training
   - Penn Treebank (PTB) for language modeling
4. Runs actual training through:
   - `apps.simple_ml.train_cifar10` / `evaluate_cifar10`
   - `apps.simple_ml.train_ptb` / `evaluate_ptb`
5. Runs small inference demos:
   - CIFAR-10 single-image prediction
   - PTB next-token autoregressive generation

## Notes before running

- Run this notebook from the repository root: `Needle_CMU_DLSys_HW`.
- If your C++ backend cannot compile in your current environment, the notebook will still run with numpy backend.
- CIFAR training uses a small subset by default for faster iteration. You can scale it up later.
- PTB training is configured for quick smoke-training, not final benchmark quality.


## 0) Environment and Project Setup

This section validates the working directory and sets Python import paths.
If you launch Jupyter from somewhere else, imports like `from apps.models import ...` may fail.

In [1]:
from pathlib import Path
import os
import sys
import random
import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    # Fallback when tqdm is not installed; keeps notebook runnable.
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "apps").exists():
    raise RuntimeError("Please open this notebook from the Needle_CMU_DLSys_HW repository root.")

# Ensure local project modules are importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "python") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "python"))

# Set deterministic seeds for reproducibility in quick experiments.
np.random.seed(0)
random.seed(0)

print("PROJECT_ROOT:", PROJECT_ROOT)

def _safe_set_postfix(progress_bar, values):
    if hasattr(progress_bar, "set_postfix"):
        progress_bar.set_postfix(values)


PROJECT_ROOT: /home/uceeanz/Needle_CMU_DLSys_HW


## 1) Build Custom Backends and Select Runtime Backend

This section is split into three parts:

1. **Configuration cell**: choose build/runtime options.
2. **Build cell**: compile your custom backend(s), including CUDA if available.
3. **Runtime selection cell**: choose `np`, `nd-cpu`, or `nd-cuda` before importing Needle.

### Why this order matters

`NEEDLE_BACKEND` is read during module import. So backend selection must happen **before** `import needle`.

### Runtime modes

- `np`: numpy backend only
- `nd-cpu`: custom Needle backend + CPU device
- `nd-cuda`: custom Needle backend + CUDA device (falls back to CPU if allowed and CUDA is unavailable)


In [2]:
# ===== Backend Build / Runtime Configuration =====

# Build options
BUILD_CUSTOM_BACKEND = True
FORCE_RECONFIGURE = False   # Re-run CMake configure even if build dir exists
BUILD_WITH_CUDA = True      # If False, CUDA detection is disabled at CMake configure time

# Runtime selection options: "np", "nd-cpu", "nd-cuda"
RUNTIME_BACKEND = "nd-cuda"
ALLOW_RUNTIME_FALLBACK = True  # If nd-cuda is unavailable, fallback to nd-cpu

print("BUILD_CUSTOM_BACKEND =", BUILD_CUSTOM_BACKEND)
print("FORCE_RECONFIGURE   =", FORCE_RECONFIGURE)
print("BUILD_WITH_CUDA     =", BUILD_WITH_CUDA)
print("RUNTIME_BACKEND     =", RUNTIME_BACKEND)
print("ALLOW_RUNTIME_FALLBACK =", ALLOW_RUNTIME_FALLBACK)


BUILD_CUSTOM_BACKEND = True
FORCE_RECONFIGURE   = False
BUILD_WITH_CUDA     = True
RUNTIME_BACKEND     = nd-cuda
ALLOW_RUNTIME_FALLBACK = True


In [4]:
import subprocess

build_dir = PROJECT_ROOT / "build_notebook"
cpu_so = PROJECT_ROOT / "python/needle/backend_ndarray/ndarray_backend_cpu.so"
cuda_so = PROJECT_ROOT / "python/needle/backend_ndarray/ndarray_backend_cuda.so"

if BUILD_CUSTOM_BACKEND:
    configure_cmd = ["cmake", "-S", str(PROJECT_ROOT), "-B", str(build_dir)]
    if not BUILD_WITH_CUDA:
        configure_cmd.append("-DCMAKE_DISABLE_FIND_PACKAGE_CUDA=ON")

    if FORCE_RECONFIGURE or not build_dir.exists():
        print("Configuring CMake...")
        print(" ".join(configure_cmd))
        subprocess.run(configure_cmd, check=True)

    print("Building backend modules...")
    subprocess.run(["cmake", "--build", str(build_dir), "-j"], check=True)

cpu_sos = list((PROJECT_ROOT / "python/needle/backend_ndarray").glob("ndarray_backend_cpu*.so"))
cuda_sos = list((PROJECT_ROOT / "python/needle/backend_ndarray").glob("ndarray_backend_cuda*.so"))

print("CPU backend module exists :", len(cpu_sos) > 0, cpu_sos[0] if cpu_sos else None)
print("CUDA backend module exists:", len(cuda_sos) > 0, cuda_sos[0] if cuda_sos else None)


Building backend modules...
[ 50%] Built target ndarray_backend_cpu
[100%] Built target ndarray_backend_cuda
CPU backend module exists : True /home/uceeanz/Needle_CMU_DLSys_HW/python/needle/backend_ndarray/ndarray_backend_cpu.cpython-312-x86_64-linux-gnu.so
CUDA backend module exists: True /home/uceeanz/Needle_CMU_DLSys_HW/python/needle/backend_ndarray/ndarray_backend_cuda.cpython-312-x86_64-linux-gnu.so


In [5]:
runtime_mode = RUNTIME_BACKEND.strip().lower()
if runtime_mode not in {"np", "nd-cpu", "nd-cuda"}:
    raise ValueError("RUNTIME_BACKEND must be one of: 'np', 'nd-cpu', 'nd-cuda'")

# Backend type is fixed at import time; restart kernel before switching modes.
if "needle" in sys.modules:
    raise RuntimeError(
        "needle is already imported in this kernel. Restart kernel after changing RUNTIME_BACKEND."
    )

# Set import-time backend flag before importing needle.
if runtime_mode == "np":
    os.environ["NEEDLE_BACKEND"] = "np"
else:
    os.environ["NEEDLE_BACKEND"] = "nd"

import needle as ndl
import needle.nn as nn
from apps.models import ResNet9, LanguageModel
from apps.simple_ml import train_cifar10, evaluate_cifar10, train_ptb, evaluate_ptb

if runtime_mode == "np":
    device = ndl.cpu()
    ACTIVE_RUNTIME_BACKEND = "np"
elif runtime_mode == "nd-cpu":
    device = ndl.cpu()
    ACTIVE_RUNTIME_BACKEND = "nd-cpu"
else:
    cuda_device = ndl.cuda()
    if cuda_device.enabled():
        device = cuda_device
        ACTIVE_RUNTIME_BACKEND = "nd-cuda"
    elif ALLOW_RUNTIME_FALLBACK:
        print("CUDA backend is not available at runtime. Falling back to nd-cpu.")
        device = ndl.cpu()
        ACTIVE_RUNTIME_BACKEND = "nd-cpu"
    else:
        raise RuntimeError("Requested nd-cuda, but CUDA backend is unavailable.")

print("NEEDLE_BACKEND env =", os.environ.get("NEEDLE_BACKEND"))
print("ACTIVE_RUNTIME_BACKEND =", ACTIVE_RUNTIME_BACKEND)
print("Device =", device)
if os.environ.get("NEEDLE_BACKEND") == "nd":
    print("All devices reported by Needle:", ndl.all_devices())


Using needle backend
NEEDLE_BACKEND env = nd
ACTIVE_RUNTIME_BACKEND = nd-cuda
Device = cuda()
All devices reported by Needle: [cpu(), cuda(), cpu_numpy()]


## 2) Dataset Download Utilities

This section downloads two datasets into `data/` under the repository root:

- **CIFAR-10** (vision classification)
- **Penn Treebank (PTB)** (language modeling corpus)

The helper is idempotent: existing files are reused and not downloaded again.

In [6]:
import tarfile
import urllib.request

data_root = PROJECT_ROOT / "data"
data_root.mkdir(exist_ok=True)

def download_file(url: str, dst: Path):
    if dst.exists():
        print("Skip (already exists):", dst)
        return
    print("Downloading:", url)
    urllib.request.urlretrieve(url, dst)

# CIFAR-10 archive and extraction target
cifar_tar = data_root / "cifar-10-python.tar.gz"
cifar_dir = data_root / "cifar-10-batches-py"
if not cifar_dir.exists():
    download_file("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", cifar_tar)
    print("Extracting:", cifar_tar)
    with tarfile.open(cifar_tar, "r:gz") as tf:
        tf.extractall(path=data_root)
else:
    print("Skip extraction (already exists):", cifar_dir)

# PTB text files expected by ndl.data.Corpus
ptb_dir = data_root / "ptb"
ptb_dir.mkdir(exist_ok=True)
ptb_urls = {
    "train.txt": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.train.txt",
    "test.txt": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt",
}
for name, url in ptb_urls.items():
    download_file(url, ptb_dir / name)

print("\nData root:", data_root)
print("CIFAR path:", cifar_dir)
print("PTB path:", ptb_dir)

Skip extraction (already exists): /home/uceeanz/Needle_CMU_DLSys_HW/data/cifar-10-batches-py
Skip (already exists): /home/uceeanz/Needle_CMU_DLSys_HW/data/ptb/train.txt
Skip (already exists): /home/uceeanz/Needle_CMU_DLSys_HW/data/ptb/test.txt

Data root: /home/uceeanz/Needle_CMU_DLSys_HW/data
CIFAR path: /home/uceeanz/Needle_CMU_DLSys_HW/data/cifar-10-batches-py
PTB path: /home/uceeanz/Needle_CMU_DLSys_HW/data/ptb


## 3) Vision Pipeline: CIFAR-10 Training and Inference

### Why use a subset first?

Full CIFAR-10 training can be slow in a notebook loop, especially without GPU acceleration.
For an interactive entrypoint, it is usually better to confirm that your full data pipeline works end-to-end on a smaller subset, then scale up.

Default subset in this notebook:
- Train: 5,000 samples
- Test: 1,000 samples

You can increase these values once your environment is stable.

In [7]:
CIFAR_TRAIN_LIMIT = 5000
CIFAR_TEST_LIMIT = 1000

train_ds = ndl.data.CIFAR10Dataset(str(cifar_dir), train=True)
test_ds = ndl.data.CIFAR10Dataset(str(cifar_dir), train=False)

# Truncate to a small subset for fast experimentation.
train_ds.X = train_ds.X[:CIFAR_TRAIN_LIMIT]
train_ds.y = train_ds.y[:CIFAR_TRAIN_LIMIT]
test_ds.X = test_ds.X[:CIFAR_TEST_LIMIT]
test_ds.y = test_ds.y[:CIFAR_TEST_LIMIT]

train_loader = ndl.data.DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = ndl.data.DataLoader(test_ds, batch_size=64, shuffle=False)

print("CIFAR train subset size:", len(train_ds))
print("CIFAR test subset size :", len(test_ds))

/home/uceeanz/Needle_CMU_DLSys_HW/python/needle/data/datasets/cifar10_dataset.py:46: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data_dict = pickle.load(f, encoding="bytes")


CIFAR train subset size: 5000
CIFAR test subset size : 1000


### Model Selection Strategy

- If backend is `nd`, we use `ResNet9` from `apps.models` directly.
- If backend is `np`, we switch to a tiny MLP to keep the notebook executable.

Reason: in some local setups, Conv-heavy paths are backend-sensitive, and this fallback keeps your runbook useful even when C++ extension build is unavailable.

In [8]:
if ACTIVE_RUNTIME_BACKEND.startswith("nd"):
    cifar_model = ResNet9(device=device, dtype="float32")
    print("Model: ResNet9 (apps.models)")
else:
    print("Model: TinyMLP fallback (numpy backend mode)")

    class TinyMLP(nn.Module):
        def __init__(self, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Flatten(),
                nn.Linear(3 * 32 * 32, hidden),
                nn.ReLU(),
                nn.Linear(hidden, 10),
            )

        def forward(self, x):
            return self.net(x)

    cifar_model = TinyMLP()


Model: ResNet9 (apps.models)


In [9]:
CIFAR_EPOCHS = 10

epoch_bar = tqdm(range(1, CIFAR_EPOCHS + 1), desc="CIFAR training", unit="epoch")
for epoch in epoch_bar:
    train_acc, train_loss = train_cifar10(
        cifar_model,
        train_loader,
        n_epochs=1,
        optimizer=ndl.optim.Adam,
        lr=1e-3,
        weight_decay=1e-4,
        show_progress=True,
    )

    test_acc, test_loss = evaluate_cifar10(cifar_model, test_loader, show_progress=True)

    _safe_set_postfix(epoch_bar, {
        "train_acc": f"{train_acc:.4f}",
        "train_loss": f"{train_loss:.4f}",
        "test_acc": f"{test_acc:.4f}",
        "test_loss": f"{test_loss:.4f}",
    })

    print(
        f"[CIFAR][Epoch {epoch}/{CIFAR_EPOCHS}] "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f}, "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


CIFAR training:   0%|          | 0/10 [00:00<?, ?epoch/s]

CIFAR train: 0batch [00:00, ?batch/s]

CIFAR eval: 0batch [00:00, ?batch/s]

[CIFAR][Epoch 1/10] train_acc=0.2510, train_loss=2.1414, test_acc=0.2990, test_loss=1.9825


CIFAR train: 0batch [00:00, ?batch/s]

KeyboardInterrupt: 

In [ ]:
# CIFAR inference demo with a progress bar over multiple samples
cifar_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

CIFAR_INFER_SAMPLES = 128
num_eval = min(CIFAR_INFER_SAMPLES, len(test_ds))

correct = 0
examples = []

for i in tqdm(range(num_eval), desc="CIFAR inference", unit="sample"):
    x_np, y_np = test_ds[i]
    x = ndl.Tensor(x_np[None, ...], requires_grad=False)
    logits = cifar_model(x).numpy()[0]
    pred = int(np.argmax(logits))

    y_int = int(y_np)
    correct += int(pred == y_int)

    if len(examples) < 5:
        examples.append((i, y_int, pred))

print(f"CIFAR inference accuracy on {num_eval} samples: {correct / num_eval:.4f}")
for idx, y_true, y_pred in examples:
    print(
        f"sample[{idx}] true={y_true} ({cifar_classes[y_true]}), "
        f"pred={y_pred} ({cifar_classes[y_pred]})"
    )


## 4) Language Pipeline: PTB Training and Generation

This section now supports three language-model backbones:
- `rnn` (via `apps.models.LanguageModel`)
- `lstm` (via `apps.models.LanguageModel`)
- `transformer` (via a notebook-side wrapper that uses existing Needle `nn.Transformer`)

Why a wrapper for Transformer?
`train_ptb` / `evaluate_ptb` expect the model API `forward(x, h) -> (logits, h)`.
The wrapper keeps this API while internally using token embedding + Transformer + LM head.

For fast experimentation, defaults are small and conservative. Increase model size/epochs later for quality runs.

In [12]:
PTB_MAX_LINES = 500 # set to None for full corpus

ptb_corpus = ndl.data.Corpus(str(ptb_dir), max_lines=PTB_MAX_LINES)
vocab_size = len(ptb_corpus.dictionary)

ptb_train = ndl.data.batchify(ptb_corpus.train, batch_size=16, device=device, dtype="float32")
ptb_test = ndl.data.batchify(ptb_corpus.test, batch_size=16, device=device, dtype="float32")

print("PTB vocab_size:", vocab_size)
print("PTB train shape:", ptb_train.shape)
print("PTB test shape :", ptb_test.shape)

PTB vocab_size: 3318
PTB train shape: (688, 16)
PTB test shape : (688, 16)


In [15]:
PTB_MODEL_TYPE = "transformer"  # choose from: "rnn", "lstm", "transformer"
PTB_SEQ_LEN = 10
PTB_EPOCHS = 2

if PTB_MODEL_TYPE in {"rnn", "lstm"}:
    lm = LanguageModel(
        embedding_size=64,
        output_size=vocab_size,
        hidden_size=128,
        num_layers=1,
        seq_model=PTB_MODEL_TYPE,
        seq_len=PTB_SEQ_LEN,
        device=device,
        dtype="float32",
    )
    lm.is_recurrent = True

    ptb_optimizer = ndl.optim.SGD
    ptb_lr = 1.0
    ptb_weight_decay = 0.0

elif PTB_MODEL_TYPE == "transformer":
    class TransformerLanguageModel(nn.Module):
        """Adapter that matches train_ptb API: forward(x, h) -> (logits, h)."""

        def __init__(
            self,
            vocab_size,
            embedding_size=128,
            hidden_size=256,
            num_layers=2,
            num_head=4,
            dim_head=32,
            dropout=0.1,
            max_seq_len=512,
            device=None,
            dtype="float32",
        ):
            super().__init__()
            self.is_recurrent = False
            self.embedding = nn.Embedding(vocab_size, embedding_size, device=device, dtype=dtype)
            self.backbone = nn.Transformer(
                embedding_size=embedding_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                num_head=num_head,
                dim_head=dim_head,
                dropout=dropout,
                causal=True,
                device=device,
                dtype=dtype,
                batch_first=False,
                sequence_len=max_seq_len,
            )
            self.lm_head = nn.Linear(embedding_size, vocab_size, device=device, dtype=dtype)

        def forward(self, x, h=None):
            # x: (seq_len, bs) token ids
            emb = self.embedding(x)
            out, _ = self.backbone(emb, h)
            seq_len, bs, dim = out.shape
            logits = self.lm_head(out.reshape((seq_len * bs, dim)))
            return logits, None

    lm = TransformerLanguageModel(
        vocab_size=vocab_size,
        embedding_size=128,
        hidden_size=256,
        num_layers=2,
        num_head=4,
        dim_head=32,
        dropout=0.1,
        max_seq_len=max(PTB_SEQ_LEN, 512),
        device=device,
        dtype="float32",
    )

    ptb_optimizer = ndl.optim.Adam
    ptb_lr = 3e-4
    ptb_weight_decay = 0.0

else:
    raise ValueError("PTB_MODEL_TYPE must be one of: 'rnn', 'lstm', 'transformer'")

print("PTB model type:", PTB_MODEL_TYPE)

epoch_bar = tqdm(range(1, PTB_EPOCHS + 1), desc=f"PTB training ({PTB_MODEL_TYPE})", unit="epoch")
for epoch in epoch_bar:
    lm_train_acc, lm_train_loss = train_ptb(
        lm,
        ptb_train,
        seq_len=PTB_SEQ_LEN,
        n_epochs=1,
        optimizer=ptb_optimizer,
        lr=ptb_lr,
        weight_decay=ptb_weight_decay,
        device=device,
        dtype="float32",
        show_progress=True,
    )

    lm_test_acc, lm_test_loss = evaluate_ptb(
        lm,
        ptb_test,
        seq_len=PTB_SEQ_LEN,
        device=device,
        dtype="float32",
        show_progress=True,
    )

    _safe_set_postfix(epoch_bar, {
        "train_acc": f"{lm_train_acc:.4f}",
        "train_loss": f"{lm_train_loss:.4f}",
        "test_acc": f"{lm_test_acc:.4f}",
        "test_loss": f"{lm_test_loss:.4f}",
    })

    print(
        f"[PTB][{PTB_MODEL_TYPE}][Epoch {epoch}/{PTB_EPOCHS}] "
        f"train_acc={lm_train_acc:.4f}, train_loss={lm_train_loss:.4f}, "
        f"test_acc={lm_test_acc:.4f}, test_loss={lm_test_loss:.4f}"
    )


PTB model type: transformer


PTB training (transformer):   0%|          | 0/2 [00:00<?, ?epoch/s]

PTB train:   0%|          | 0/69 [00:00<?, ?step/s]

KeyboardInterrupt: 

In [ ]:
def generate_text(model, corpus, start_word, max_new_tokens=30, show_progress=True):
    """Greedy autoregressive generation for recurrent and transformer models."""
    word2idx = corpus.dictionary.word2idx
    idx2word = corpus.dictionary.idx2word

    if start_word not in word2idx:
        start_word = idx2word[0]

    model.eval()
    token_ids = [word2idx[start_word]]
    tokens = [start_word]

    is_recurrent = getattr(model, "is_recurrent", True)
    h = None

    token_iter = range(max_new_tokens)
    if show_progress:
        token_iter = tqdm(token_iter, desc="Text generation", unit="token", leave=False)

    for _ in token_iter:
        if is_recurrent:
            # Recurrent models consume one token with carried hidden state.
            x = ndl.Tensor(np.array([[token_ids[-1]]], dtype=np.int64), requires_grad=False)
            logits, h = model(x, h)
        else:
            # Transformer is stateless here; feed full context each step.
            x = ndl.Tensor(np.array(token_ids, dtype=np.int64).reshape((-1, 1)), requires_grad=False)
            logits, _ = model(x, None)

        next_id = int(np.argmax(logits.numpy()[-1]))
        token_ids.append(next_id)
        tokens.append(idx2word[next_id])

    return " ".join(tokens)

seed = "the" if "the" in ptb_corpus.dictionary.word2idx else ptb_corpus.dictionary.idx2word[0]
print(generate_text(lm, ptb_corpus, start_word=seed, max_new_tokens=30, show_progress=True))


## 5) Suggested Next Steps

If this notebook runs end-to-end successfully, typical next improvements are:

1. Increase CIFAR subset to full train/test and run more epochs.
2. Keep only `nd` backend path once your local build environment is stable.
3. Add checkpoint saving/loading for both models.
4. Add richer metrics (per-class accuracy, confusion matrix, perplexity for PTB).
5. Add configurable hyperparameter cells at the top for reproducible experiments.